In [ ]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [ ]:
df_HV = pd.read_csv('valid_HV_24h_tol2_drift_pairs_with_errors.csv')

In [ ]:
df_HV.columns

In [ ]:
import pandas as pd
from pathlib import Path

# Ensure buoy times are datetime
df_HV["t0"] = pd.to_datetime(df_HV["t0"])
df_HV["t1"] = pd.to_datetime(df_HV["t1"])

# Function to extract SAR timestamp from TIFF path
def extract_sar_time(path):
    filename = Path(path).stem  # e.g. '20190424T0653'
    return pd.to_datetime(filename, format="%Y%m%dT%H%M")

# Extract SAR timestamps
df_HV["sar_t0"] = df_HV["tiff0_path"].apply(extract_sar_time)
df_HV["sar_t1"] = df_HV["tiff1_path"].apply(extract_sar_time)

# Compute time differences in hours (signed)
df_HV["buoy_SAR_dt0_hours"] = (
    (df_HV["t0"] - df_HV["sar_t0"]).dt.total_seconds() / 3600
)

df_HV["buoy_SAR_dt1_hours"] = (
    (df_HV["t1"] - df_HV["sar_t1"]).dt.total_seconds() / 3600
)

# Absolute differences
df_HV["buoy_SAR_dt0_abs_hours"] = df_HV["buoy_SAR_dt0_hours"].abs()
df_HV["buoy_SAR_dt1_abs_hours"] = df_HV["buoy_SAR_dt1_hours"].abs()

In [ ]:
df_HV = df_HV[(df_HV["buoy_SAR_dt0_abs_hours"] <= 0.5) & (df_HV["buoy_SAR_dt1_abs_hours"] <= 0.5)]
df_HV = df_HV[df_HV['n_pm']>400]

In [ ]:
df_HV['rel_endpoint_err'].notna().sum()

In [ ]:
df_HV = df_HV[df_HV['rel_endpoint_err'].notna()]

In [ ]:
# df_HV[df_HV['endpoint_err_m']> 5000].iloc[6]

In [ ]:
# df_HV = df_HV[df_HV['rel_endpoint_err'] < 10]

In [ ]:
len(df_HV)

In [ ]:
df_HV['rel_endpoint_err'].describe()

In [ ]:
df_HV['endpoint_err_m'].describe()

In [ ]:
df_HV['angle_error_deg'].describe()

In [ ]:
df_HV['n_ft'].describe()    

In [ ]:
error_cols = [
    "abs_mag_error_m",
    "angle_error_deg",
    "endpoint_err_m",
    "rel_endpoint_err",
]


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, col in zip(axes, error_cols):
    sns.violinplot(
        y=df_HV[col],
        inner=None,
        cut=0,
        ax=ax
    )
    sns.boxplot(
        y=df_HV[col],
        width=0.2,
        boxprops={"facecolor":"white"},
        showfliers=True,
        ax=ax
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()


In [ ]:
cols_corr = ["rel_endpoint_err", "n_ft", "n_pm"]
df_corr = df_HV[cols_corr].dropna()


In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

pairs = [("n_ft", axes[0]), ("n_pm", axes[1])]

for xcol, ax in pairs:
    x = df_corr[xcol]
    y = df_corr["rel_endpoint_err"]

    r = np.corrcoef(x, y)[0, 1]

    ax.scatter(x, y, alpha=0.6)
    ax.set_xlabel(xcol)
    ax.set_title(f"rel_endpoint_err vs {xcol}\nPearson r = {r:.2f}")
    ax.grid(True)

axes[0].set_ylabel("rel_endpoint_err")
plt.tight_layout()
plt.show()


In [ ]:
df_corr.corr()


In [ ]:
sns.heatmap(df_corr.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()
